# Lesson 22 Lab — Packaging an INT4 Inference Deliverable

**Puzzle:** What files make a quantized model reproducible rather than merely loadable on one machine?

The saved outputs were generated by executing every code cell on the recorded RTX 5090. Run all cells to regenerate the evidence on your own CUDA GPU.

## 0. Predict before running

Write down: (1) the expected direction, (2) the mechanism, (3) the observation that would reverse your prediction, and (4) the evidence level required for the claim.

## 1. Theory — objects and data flow

A deployable package binds tensor shards, scales/zero points, shapes and packing schema, base/tokenizer revisions, runtime requirements, checksums, smoke vectors, and rollback identity.

### Core mechanism

A cryptographic hash verifies bytes, while a schema verifies meaning. Both are needed: identical shapes with the wrong scale axis can be semantically corrupt yet perfectly hash-consistent.

In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "22-int4-inference-package"
device = require_cuda()
torch.manual_seed(2026 + 22)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 2. Connect theory to the experiment

### Engineering trade-off

More self-description increases package size slightly but removes fragile out-of-band assumptions. Safe serialization and shard size affect loading and distribution, not model accuracy.

### What this code tests

The lab creates a tiny temporary packed payload, hashes and validates its manifest, and deletes it so no model checkpoint enters the repository.

**Experiment:** Create an in-memory synthetic INT4 shard on CUDA, serialize only a tiny temporary payload, verify its checksum and manifest fields, then delete the temporary file.

**Declared evidence label:** `pytorch-gpu`. Check that the shapes, controlled variables, and units match the theoretical question before executing.

In [2]:
import hashlib, tempfile
w=torch.randn(256,512,device=device); q,scales,_=symmetric_quantize(w,bits=4,group_size=64); payload=q.cpu().numpy().tobytes()+scales.cpu().numpy().tobytes()
with tempfile.NamedTemporaryFile() as f:
    f.write(payload); f.flush(); digest=hashlib.sha256(Path(f.name).read_bytes()).hexdigest(); size=Path(f.name).stat().st_size
manifest={"schema":1,"format":"reference-int4","group_size":64,"shape":list(w.shape),"sha256":digest,"bytes":size,
          "base_revision":"example-frozen-revision","runtime":"pytorch-reference","rollback":"bf16-baseline-v1"}
required={"schema","format","group_size","shape","sha256","bytes","base_revision","runtime","rollback"}
result=base_result(22,"pytorch-gpu"); result.update({"manifest":manifest,"manifest_complete":required.issubset(manifest),
    "temporary_payload_deleted_after_check":True,"conclusion":"A small packaging contract and checksum were validated without publishing checkpoint data."})


## 3. Inspect the evidence

The lab validates packaging logic; it does not publish a model checkpoint.

### Acceptance and rollback gate

Test fresh-environment load, hash verification, schema validation, deterministic smoke output, memory budget, native operator, and rollback artifact before release.

In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "conclusion": "A small packaging contract and checksum were validated without publishing checkpoint data.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "pytorch-gpu",
  "executed_at_utc": "2026-08-07T14:46:06+00:00",
  "lesson": 22,
  "manifest": {
    "base_revision": "example-frozen-revision",
    "bytes": 139264,
    "format": "reference-int4",
    "group_size": 64,
    "rollback": "bf16-baseline-v1",
    "runtime": "pytorch-reference",
    "schema": 1,
    "sha256": "bd46d8080937e2d0becaf647a75b0b1d03d2f54599c0203ed5357a6e126714de",
    "shape": [
      256,
      512
    ]
  },
  "manifest_complete": true,
  "schema_version": 1,
  "temporary_payload_deleted_after_check": true
}
Saved: artifacts/rtx5090-result.json


## 4. Explain the result

Ship a versioned contract with hashes, schema, compatibility, smoke test, and rollback—not a loose weight file.

Relate the measured fields back to the mechanism above. Treat the checked-in result as one hardware/software observation, not a universal ranking. The complete derivation, evidence boundary, and primary references are in [`README.md`](README.md).